## 📦 Install WikiExtractor

We use `wikiextractor` to extract clean text from Wikipedia dumps.

First, install it via pip. We recommend using the latest version directly from GitHub.

In [ ]:
!pip install wikiextractor
!pip install git+https://github.com/attardi/wikiextractor.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.8 MB/s eta 0:00:00
  Cloning https://github.com/attardi/wikiextractor.git to /tmp/pip-req-build-4ikcboud
  Running command git clone --filter=blob:none --quiet https://github.com/attardi/wikiextractor.git /tmp/pip-req-build-4ikcboud
  Resolved https://github.com/attardi/wikiextractor.git to commit 8f1b434a80608e1e313d38d263ed7c79c9ee75a9
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


## 📄 Extract Text from Wikipedia XML Dump

We define a function to extract raw text from `<text>` tags in a Wikipedia XML dump using Python's built-in `xml.etree.ElementTree`.

- Removes formatting like `[[link|display]]`, `{{templates}}`, and `== headings ==`
- Strips newlines and joins text from multiple pages
- Limits the number of pages processed using `max_pages`

In [ ]:
import xml.etree.ElementTree as ET

def extract_wikipedia_text(xml_path, max_pages=None):
    """Extracts plain text from <text> tags in the Wikipedia XML dump."""
    extracted_text = []
    page_count = 0

    for event, elem in ET.iterparse(xml_path, events=("start", "end")):
        if event == "end" and elem.tag.endswith("text"):
            text_content = elem.text
            if text_content:
                # Basic cleanup: remove things like [[link|display]], {{templates}}, and ==
                cleaned = (
                    text_content
                    .replace('\n', ' ')  # remove newlines
                    .replace('[[', '').replace(']]', '')
                )
                extracted_text.append(cleaned)

                page_count += 1
                if max_pages and page_count >= max_pages:
                    break
            elem.clear()


    return " ".join(extracted_text)

In [ ]:
# Fix the regex issue in WikiExtractor's extract.py
import os

file_path = "/usr/local/lib/python3.11/dist-packages/wikiextractor/extract.py"

# Read and fix the line causing the regex error
with open(file_path, "r") as f:
    lines = f.readlines()

with open(file_path, "w") as f:
    for line in lines:
        if "ExtLinkBracketedRegex" in line:
            # Move inline flag (?i) to the beginning of the regex
            line = line.replace(r"(?i)", "")  # remove inline flag if not at beginning
            line = line.replace("re.compile(r'", "re.compile(r'(?i)")  # place flag at start
        f.write(line)

## 🧹 Extract Clean JSON Articles from Wikipedia Dump

We use `WikiExtractor` to process the Wikipedia XML file and extract clean text.

- Input: `simplewiki-latest-pages-articles.xml`
- Output: Folder `extracted_simplewiki` with JSON files
- Format: `--json` saves output in structured JSON format (recommended for further NLP processing)

🛠️ **Command:**

In [ ]:
!python -m wikiextractor.WikiExtractor "/content/simplewiki-latest-pages-articles.xml" -o extracted_simplewiki --json
!python --version


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.11/dist-packages/wikiextractor/WikiExtractor.py", line 66, in <module>
    from .extract import Extractor, ignoreTag, define_template, acceptedNamespaces
  File "/usr/local/lib/python3.11/dist-packages/wikiextractor/extract.py", line 378, in <module>
    ExtLinkBracketedRegex = re.compile(
                            ^^^^^^^^^^^
  File "/usr/lib/python3.11/re/__init__.py", line 227, in compile
    return _compile(pattern, flags)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/re/__init__.py", line 294, in _compile
    p = _compiler.compile(pattern, flags)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/re/_compiler.py", line 745, in compile
    p = _parser.parse(p, flags)
        ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/re/_parser.py", line 989, in parse
    p = _parse_

## 📥 Load and Extract Sample Text from Wikipedia Dump

Now we use our `extract_wikipedia_text()` function to read and process the Simple English Wikipedia XML file.

- We define the file path.
- Extract text from up to `1000` pages to keep it lightweight.
- Display a sample of the extracted text.

In [ ]:
# Set the path to your Simple English Wikipedia XML file
wiki_path = r"/content/simplewiki-latest-pages-articles.xml"

# Extract text (you can set max_pages to limit processing time)
simplewiki_text = extract_wikipedia_text(wiki_path, max_pages=1000)

# View sample
print("Sample text:\n", simplewiki_text[:1000])

NameError: name 'simplewiki_cleaned' is not defined

## 🧼 Clean Extracted Text

We now clean the extracted Wikipedia text to prepare it for modeling or further analysis.

The cleaning steps include:

- **Removing non-ASCII characters** (e.g., symbols, non-English characters)
- **Stripping Wikipedia-style headers** like `== History ==`
- **Normalizing whitespace** (removing extra spaces, newlines, tabs)
  

In [ ]:
 import re

clean_text = simplewiki_text

# Remove non-ASCII characters
clean_text = re.sub(r'[^\x00-\x7F]+', ' ', clean_text)

# Remove HTML/XML-style comments like <!-- comment -->
clean_text = re.sub(r'<!--.*?-->', ' ', clean_text, flags=re.DOTALL)

# Remove headers like "== History =="
clean_text = re.sub(r'={2,}.*?={2,}', ' ', clean_text)

# Remove HTML or XML tags like <ref>...</ref>
clean_text = re.sub(r'<[^>]+>', ' ', clean_text)

# Remove Wikipedia templates like {{cite ...}}, {{Infobox ...}}, etc.
clean_text = re.sub(r'\{\{.*?\}\}', ' ', clean_text, flags=re.DOTALL)

# Remove infobox/table-style lines starting with "|"
clean_text = re.sub(r'^\|.*$', ' ', clean_text, flags=re.MULTILINE)

# Remove inline key = value expressions (e.g., "HDI = 0.887")
clean_text = re.sub(r'\b[\w\s]+\s*=\s*[^=]+(?=\s|$)', ' ', clean_text)

# Remove remaining curly-brace or pipe blocks
clean_text = re.sub(r'\{[^{}]*\}', ' ', clean_text)
clean_text = re.sub(r'\|', ' ', clean_text)

# Replace multiple newlines or spaces with a single space
clean_text = re.sub(r'\s+', ' ', clean_text).strip()


## 💾 Save and Subset Cleaned Text

After cleaning, we can:

- **Optionally save** the full cleaned text to a `.txt` file for later use.
- **Extract a subset** (e.g., first 500,000 characters) to keep experiments lightweight or for faster training/testing.

📁 Output file: `simplewiki_cleaned.txt`

In [ ]:
# Load the full text first
# Save cleaned text (optional)
with open("simplewiki_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)





In [ ]:
from collections import defaultdict, Counter
import random

class LZInductionModel:
    def __init__(self, max_context=50):
        self.max_context = max_context
        self.memory = defaultdict(Counter)

    def train(self, text, verbose=True, log_every=100000):
        total = len(text)
        for i in range(len(text) - 1):
            for ctx_len in range(1, self.max_context + 1):
                if i - ctx_len < 0:
                    break
                context = text[i - ctx_len:i]
                next_char = text[i]
                self.memory[context][next_char] += 1
        if verbose and i % log_every == 0 and i > 0:
            print(f"[{i}/{total}] characters processed. Unique contexts: {len(self.memory)}")
    def predict_next(self, context, temperature=1.0):
        for ctx_len in range(min(self.max_context, len(context)), 0, -1):
            sub_ctx = context[-ctx_len:]
            if sub_ctx in self.memory:
                options = self.memory[sub_ctx]
                chars, freqs = zip(*options.items())
                freqs = [f ** (1 / temperature) for f in freqs]
                total = sum(freqs)
                probs = [f / total for f in freqs]
                return random.choices(chars, probs)[0]
        return random.choice("abcdefghijklmnopqrstuvwxyz ")  # fallback

    def generate(self, start, length=100, temperature=1.0):
        result = start
        for _ in range(length):
            next_char = self.predict_next(result, temperature)
            result += next_char
        return result

In [ ]:
lz_model = LZInductionModel(max_context=40)

chunk_size = 50000  # 50 KB chunks
for i in range(0, len(clean_text), chunk_size):
         chunk = clean_text[i:i + chunk_size]
         lz_model.train(chunk)

In [ ]:
# Function to generate all left-aligned prefixes
def generate_prefixes(sentence):
    tokens = sentence.strip().split()
    prefixes = [' '.join(tokens[:i]) for i in range(1, len(tokens)+1)]
    return prefixes

# Input start sentence
start_sentence = "The capital of France is "
prefixes = generate_prefixes(start_sentence)

# Predict next word for each prefix using LZ model
print("Predicted next words for each prefix:")
for prefix in prefixes:
    try:
        next_word =  lz_model.generate(prefix, length=50)
        print(f"{prefix!r} → {next_word!r}")
    except Exception as e:
        print(f"{prefix!r} → Error: {str(e)}")


Predicted next words for each prefix:
'The' → 'The Galactic halo extends outward but is limited in s'
'The capital' → 'The capital is Sacramento, California Sacramento. The states '
'The capital of' → 'The capital of the country is Washington, DC, and the capital of'
'The capital of France' → 'The capital of France. At that time, France had the Demographics of Fra'
'The capital of France is' → 'The capital of France is a semi-presidential system determined by the cons'


In [ ]:
import nltk

# 🔁 Re-download using updated tagger name
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger_eng')  # <- new name
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
# 📦 Import and download nltk resources
import nltk
from nltk.corpus import wordnet as wn
from nltk import pos_tag, word_tokenize


# 🔧 Helper: Simplify POS tag
def simplify_pos(tag):
    if tag.startswith('J'):
        return 'ADJ'
    elif tag.startswith('V'):
        return 'VERB'
    elif tag.startswith('N'):
        return 'NOUN'
    elif tag.startswith('R'):
        return 'ADVERB'
    elif tag == 'MD':
        return 'MODAL'
    elif tag == 'DT':
        return 'DET'
    elif tag in ['IN', 'TO']:
        return 'PREP'
    elif tag == 'CC':
        return 'CONJ'
    elif tag in ['PRP', 'PRP$']:
        return 'PRON'
    else:
        return None

# 🔧 Helper: Get lemma using WordNet
def get_lemma(word, pos=None):
    try:
        synsets = wn.synsets(word, pos=pos)
        if synsets:
            return synsets[0].lemmas()[0].name()
    except:
        pass
    return word.lower()

# 🚀 Main function: Build lexicon from outputs
def generate_lexicon_from_output(predicted_outputs):
    lexicon = {}

    for prefix, text in predicted_outputs.items():
        tokens = text.split()  #(text)
        from nltk.tag import pos_tag_sents
        tagged = pos_tag_sents([tokens])[0]

        for word, pos in tagged:
            simplified = simplify_pos(pos)
            if simplified:
                wn_pos = {
                    'NOUN': wn.NOUN,
                    'VERB': wn.VERB,
                    'ADJ': wn.ADJ,
                    'ADVERB': wn.ADV
                }.get(simplified, None)

                lemma = get_lemma(word, wn_pos)

                features = {}
                if simplified == "NOUN":
                    features["number"] = "plur" if word.endswith("s") else "sing"
                elif simplified == "VERB":
                    if pos == 'VBG':
                        features["tense"] = "participle"
                    elif pos == 'VBD':
                        features["tense"] = "past"
                    else:
                        features["tense"] = "present"

                lexicon[word] = {
                    "lemma": lemma,
                    "pos": simplified,
                    "features": features
                }

    return lexicon

# 📘 Test Example
predicted_outputs = {
  "The": "The Galactic halo extends outward but is limited in s",
  "The capital": "The capital is Sacramento, California Sacramento. The states ",
  "The capital of": "The capital of the country is Washington, DC, and the capital of",
  "The capital of France": "The capital of France. At that time, France had the Demographics of Fra",
  "The capital of France is": "The capital of France is a semi-presidential system determined by the cons"
}

# 🔍 Generate and display lexicon
lexicon = generate_lexicon_from_output(predicted_outputs)

import pprint
pprint.pprint(lexicon)


{'At': {'features': {}, 'lemma': 'astatine', 'pos': 'PREP'},
 'California': {'features': {'number': 'sing'},
                'lemma': 'California',
                'pos': 'NOUN'},
 'DC,': {'features': {'number': 'sing'}, 'lemma': 'dc,', 'pos': 'NOUN'},
 'Demographics': {'features': {'number': 'plur'},
                  'lemma': 'demographic',
                  'pos': 'NOUN'},
 'Fra': {'features': {'number': 'sing'}, 'lemma': 'fra', 'pos': 'NOUN'},
 'France': {'features': {'number': 'sing'}, 'lemma': 'France', 'pos': 'NOUN'},
 'France.': {'features': {'number': 'sing'}, 'lemma': 'france.', 'pos': 'NOUN'},
 'Galactic': {'features': {'number': 'sing'},
              'lemma': 'galactic',
              'pos': 'NOUN'},
 'Sacramento,': {'features': {'number': 'sing'},
                 'lemma': 'sacramento,',
                 'pos': 'NOUN'},
 'Sacramento.': {'features': {'number': 'sing'},
                 'lemma': 'sacramento.',
                 'pos': 'NOUN'},
 'The': {'features': {}, 'lemma

In [ ]:
!pip install -U spacy
!python -m spacy download en_core_web_sm



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 56.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def get_last_pos_tag_spacy(phrase):
    doc = nlp(phrase)
    last_token = doc[-1]
    return last_token.text, last_token.tag_  # tag_ gives Penn Treebank POS

word, pos = get_last_pos_tag_spacy(start_sentence)
print(f"Last word: {word}, POS: {pos}")

Last word: is, POS: VBZ


In [ ]:
def infer_expected_pos_from_last_tag(last_pos):
    """
    Given the POS tag of the last word in a phrase,
    return a list of expected POS tags for the next word.
    """

    # Define basic grammar rules for next expected POS
    if last_pos in {'MD'}:
        # Modal verb (e.g., "can", "should") → expect base verb (VB)
        return ['VB']

    elif last_pos in {'VB', 'VBP', 'VBZ'}:
        # Base/formal verbs → expect direct object or modifier
        return ['DT', 'NN', 'NNS', 'NNP', 'PRP$', 'JJ', 'RB']

    elif last_pos in {'VBD', 'VBN'}:
        # Past tense or past participle → likely continue with adverb or object
        return ['RB', 'DT', 'NN', 'NNS', 'IN']

    elif last_pos in {'JJ'}:
        # Adjective → expect noun
        return ['NN', 'NNS', 'NNP']

    elif last_pos in {'DT', 'PRP$'}:
        # Determiner or possessive → expect noun
        return ['JJ', 'NN', 'NNS', 'NNP']

    elif last_pos in {'NN', 'NNS', 'NNP'}:
        # Noun → could continue with verb, preposition, or punctuation
        return ['VB', 'VBD', 'IN', 'CC']

    elif last_pos in {'IN'}:
        # Preposition → expect determiner or noun
        return ['DT', 'NN', 'NNS', 'PRP$', 'JJ']

    elif last_pos in {'RB'}:
        # Adverb → expect verb or adjective
        return ['VB', 'JJ']

    else:
        # Default to noun or determiner if unsure
        return ['DT', 'NN', 'JJ']


In [ ]:
last_pos = {pos}
expec_pos = print(infer_expected_pos_from_last_tag(last_pos))



['DT', 'NN', 'JJ']


In [ ]:
def map_nltk_to_custom_pos(pos_list):
    """
    Convert a list of NLTK-style POS tags to the lexicon's custom POS tag format.
    Example: ['DT', 'NN', 'JJ'] → ['DET', 'NOUN', 'ADJ']
    """
    mapping = {
        'NN': 'NOUN',
        'NNS': 'NOUN',
        'NNP': 'NOUN',
        'VB': 'VERB',
        'VBD': 'VERB',
        'VBG': 'VERB',
        'VBN': 'VERB',
        'VBP': 'VERB',
        'VBZ': 'VERB',
        'JJ': 'ADJ',
        'JJR': 'ADJ',
        'JJS': 'ADJ',
        'RB': 'ADVERB',
        'RBR': 'ADVERB',
        'RBS': 'ADVERB',
        'DT': 'DET',
        'PRP': 'PRON',
        'PRP$': 'PRON',
        'IN': 'PREP',
        'TO': 'PREP',
        'MD': 'MODAL',
        'CC': 'CONJ'
    }

    return list({mapping.get(pos, 'NOUN') for pos in pos_list})  # fallback to 'NOUN'

# Example usage:
nltk_pos_list = ['DT', 'NN', 'JJ']
custom_pos_list = map_nltk_to_custom_pos(nltk_pos_list)
print(custom_pos_list)

['NOUN', 'ADJ', 'DET']


In [ ]:
def next_word_from_lexicon(prefix, lexicon):
    """
    prefix: str, e.g. "AI can change"
    lexicon: dict, e.g. { 'AI': {'features': {'number': 'sing'},
        'lemma': 'Army_Intelligence',
        'pos': 'NOUN'}, ... }

    Returns possible next words filtered by grammar rules and lexicon.
    """

    # Simple POS expectation after "AI can change":
    # modal + base verb → expect determiner (DT) or noun (NN)
    expected_pos = ['NOUN', 'ADJ', 'DET']

    # Filter lexicon for words with expected POS
    candidates = [word for word, info in lexicon.items() if info.get('pos') in expected_pos]

    return candidates

# Example lexicon snippet
lexicon_example = lexicon

# Using the function
possible_next = next_word_from_lexicon(start_sentence, lexicon_example)
print("Possible next words:", possible_next)


Possible next words: ['The', 'Galactic', 'halo', 's', 'capital', 'Sacramento,', 'California', 'Sacramento.', 'states', 'the', 'country', 'Washington,', 'DC,', 'France.', 'that', 'time,', 'France', 'Demographics', 'Fra', 'a', 'semi-presidential', 'system', 'cons']


In [ ]:
import re
import random
from collections import defaultdict, Counter

class SimpleSequenceMemoizer:
    def __init__(self):
        self.context_dict = defaultdict(Counter)

    def train(self, tokens, max_context=5):
        for i in range(len(tokens)):
            for k in range(1, max_context + 1):
                if i - k >= 0:
                    context = tuple(tokens[i - k:i])
                    self.context_dict[context][tokens[i]] += 1

    def predict_next(self, context, top_k=5):
        context = tuple(context)
        while context:
            if context in self.context_dict:
                counter = self.context_dict[context]
                return random.choices(list(counter.keys()), weights=counter.values())[0]
            context = context[1:]  # back off
        return random.choice(list(self.context_dict[()]))  # fallback to unigram

def tokenize(text):
    return re.findall(r'\w+|[^\w\s]', text)

def main():


    # Initialize and train model
    tokens = tokenize(clean_text)
    model = SimpleSequenceMemoizer()
    model.train(tokens, max_context=5)

    # Start with a seed phrase
    seed = "According to recent studies"
    context = tokenize(seed)

    generated = context[:]
    for _ in range(20):  # generate 20 tokens
        next_word = model.predict_next(generated[-5:])
        generated.append(next_word)

    print("Generated:", " ".join(generated))

if __name__ == "__main__":
    main()


Generated: According to recent studies how people think about moral issues , problems , and questions . Psychologists who have studied it include Lawrence Kohlberg
